# pydisplay + LVGL in Jupyter

Two sections below:

1. **Timer test** — runs the full `lv_test_timer_async` example (platform info, arc, tap counter).
2. **Minimal walkthrough** — build a small LVGL screen from scratch in a few cells.

**Requires:** `pip install pillow ipywidgets ipyevents`  
**Cursor/VS Code:** `jupyter.widgetScriptSources` in [`.vscode/settings.json`](../.vscode/settings.json)

Touch works on the **ipywidgets Image** below each running example (click that widget, not the cell output chrome). Restart the kernel to stop background LVGL tasks.

## Timer test

Runs [`lv_test_timer_async`](../examples/lv_test_timer_async.py) — platform labels, ticking arc, and tap counter. Good sanity check that LVGL + Jupyter are wired correctly.

In [ ]:
import lib.path
import lv_test_timer_async

# Interactive ipywidgets Image appears below — click that widget for touch.
# Runs as a background asyncio task; cell returns immediately. Restart kernel to stop.

In [ ]:
# Teardown verification — JNDisplay
# Run after the timer test, or alone after a kernel restart (board_config creates JNDisplay).
# Safe to re-run: uses a fresh JNDisplay if the board_config instance was already deinit'd.
import lib.path
from board_config import display_drv as _cfg_drv
from displaysys.jndisplay import JNDisplay

if getattr(_cfg_drv, "_deinitialized", False):
    display_drv = JNDisplay(_cfg_drv.width, _cfg_drv.height)
else:
    display_drv = _cfg_drv

assert display_drv.__class__.__name__ == "JNDisplay"
had_timer = display_drv._timer is not None

display_drv.fill(0x07E0)
display_drv.show()
display_drv.deinit()
assert display_drv._deinitialized
assert display_drv._timer is None
display_drv.deinit()  # idempotent
print(f"JNDisplay DEINIT_OK had_timer={had_timer}")


## Minimal LVGL walkthrough

Build a tiny UI step by step. **Restart the kernel**, then run the cells in this section **in order** (skip the timer test above).

| Cell | What it does |
|------|----------------|
| Start engine | Boots LVGL + `JNDisplay`; keeps the asyncio tick loop alive in the background |
| First widgets | Title, button, tap counter |
| Change from another cell | Rename the title, add a slider |
| Inspect | Print LVGL version and live widget state |

The kernel stays alive between cells, so you can tweak the UI without re-importing everything.

In [ ]:
# Cell 1 — start the LVGL engine (run once per kernel session)
import lib.path

try:
    import asyncio
except ImportError:
    import uasyncio as asyncio

from multimer.aio import run


async def _lvgl_keepalive():
    # Import here so asyncio is already running (Jupyter sets TIMER_ASYNC for us).
    import display_driver  # noqa: F401

    while True:
        await asyncio.sleep(0)


run(_lvgl_keepalive)
# Cell returns immediately; the display widget appears below. Click that image for touch.

In [ ]:
# Cell 2 — first widgets (safe to re-run: replaces the previous demo screen)
import lvgl as lv

if "_demo_scr" in globals():
    _demo_scr.delete()

_demo_scr = lv.obj()
_demo_scr.set_size(lv.pct(100), lv.pct(100))
lv.screen_load(_demo_scr)
scr = _demo_scr

title = lv.label(scr)
title.set_text("Hello from Jupyter")
title.align(lv.ALIGN.TOP_MID, 0, 12)

btn = lv.button(scr)
btn.set_size(140, 44)
btn.center()
btn_lbl = lv.label(btn)
btn_lbl.set_text("Tap me (0)")
btn_lbl.center()

_taps = 0


def _on_btn_click(_e):
    global _taps
    _taps += 1
    btn_lbl.set_text(f"Tap me ({_taps})")


btn.add_event_cb(_on_btn_click, lv.EVENT.CLICKED, None)
print("UI ready — tap the button on the display widget.")

In [ ]:
# Cell 3 — change the UI from a new cell (no engine restart needed)
import lvgl as lv

title.set_text("Updated title (cell 3)")

if "_demo_slider" in globals():
    _demo_slider.delete()

_demo_slider = lv.slider(scr)
_demo_slider.set_width(200)
_demo_slider.align(lv.ALIGN.BOTTOM_MID, 0, -24)


def _on_slider(_e):
    btn_lbl.set_text(f"Slider: {_demo_slider.get_value()}")


_demo_slider.add_event_cb(_on_slider, lv.EVENT.VALUE_CHANGED, None)
print(f"Children on screen: {scr.get_child_count()}")

In [ ]:
# Cell 4 — inspect live state
import lvgl as lv

info = {
    "lvgl": f"{lv.version_major()}.{lv.version_minor()}",
    "screen_children": scr.get_child_count(),
    "taps": _taps,
    "slider": _demo_slider.get_value(),
}
for key, value in info.items():
    print(f"{key}: {value}")

### Stopping

Background LVGL tasks keep running after the start-engine cell finishes. **Restart the kernel** to stop them (the notebook Stop button does not cancel `create_task` work).

To try the timer test or walkthrough again, restart the kernel and run only the cells for that section.